In [ ]:
# Databricks notebook sourceMAGIC %md# Goal: Convert Bronze policies to a clean, typed, deduped Silver table with sane dates and mandatory keys.## Steps### 1.Read Bronze Delta### 2.Type casting & column selection### 3.Key checks (Policy_ID, Customer_ID) → quarantine nulls### 4.Fix dates (start/end)### 5.Deduplicate by most recent start dateWrite to Silver + Register Table

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# 1) Read Bronzepol_bz = spark.read.format("delta").load(paths_bronze["policies"])display(pol_bz.limit(10))

In [ ]:
pol_bz.columns

In [ ]:
# ============ POLICIES → SILVER (merged enterprise version) ============from pyspark.sql import functions as Ffrom pyspark.sql.window import Windowfrom pyspark.sql.types import (    StructType, StructField, StringType, DateType, DoubleType, IntegerType)# 1) Select & cast (your original, business-faithful schema)pol = (pol_bz.select(        F.col("Policy_ID").cast("string"),        F.col("Customer_ID").cast("string"),        F.col("Product_Line").cast("string"),        F.col("Sum_Insured_GBP").cast("double"),        F.col("Annual_Premium_GBP").cast("double"),        F.to_date("Policy_Start_Date").alias("Policy_Start_Date"),        F.to_date("Policy_End_Date").alias("Policy_End_Date"),        F.col("Renewal_Offered_Flag").cast("int"),        F.col("Renewal_Accepted_Flag").cast("int"),        F.col("Discount_Offered_%").cast("double"),        F.col("Channel").cast("string")))# 2) Enforce schema explicitly (enterprise)policy_schema = StructType([    StructField("Policy_ID", StringType()),    StructField("Customer_ID", StringType()),    StructField("Product_Line", StringType()),    StructField("Sum_Insured_GBP", DoubleType()),    StructField("Annual_Premium_GBP", DoubleType()),    StructField("Policy_Start_Date", DateType()),    StructField("Policy_End_Date", DateType()),    StructField("Renewal_Offered_Flag", IntegerType()),    StructField("Renewal_Accepted_Flag", IntegerType()),    StructField("Discount_Offered_%", DoubleType()),    StructField("Channel", StringType())])pol = enforce_schema(pol, policy_schema)# 3) Mandatory keys → quarantine + keep OKpol_bad = pol.filter(F.col("Policy_ID").isNull() | F.col("Customer_ID").isNull())pol_ok  = pol.filter(F.col("Policy_ID").isNotNull() & F.col("Customer_ID").isNotNull())if pol_bad.take(1):    quarantine(pol_bad, "null_key_policy_or_customer", "policies")# 4) Date fix (keep your original intent)pol_ok = fix_dates(pol_ok, "Policy_Start_Date", "Policy_End_Date")# 5) Business validations / bounds (enterprise hardening)# - premium & sum insured non-negative# - discount between 0 and 100# - renewal accepted cannot be 1 if renewal not offered, etc.viol_premium = pol_ok.filter((F.col("Annual_Premium_GBP") < 0) | (F.col("Sum_Insured_GBP") < 0))if viol_premium.take(1):    quarantine(viol_premium, "negative_money_values", "policies")    pol_ok = pol_ok.exceptAll(viol_premium)viol_discount = pol_ok.filter(~F.col("Discount_Offered_%").between(0.0, 100.0))if viol_discount.take(1):    quarantine(viol_discount, "discount_out_of_bounds", "policies")    pol_ok = pol_ok.exceptAll(viol_discount)viol_renewal = pol_ok.filter((F.col("Renewal_Accepted_Flag") == 1) & (F.col("Renewal_Offered_Flag") != 1))if viol_renewal.take(1):    quarantine(viol_renewal, "renewal_inconsistent", "policies")    pol_ok = pol_ok.exceptAll(viol_renewal)# 6) Deduplicate (keep most recent Policy_Start_Date per Policy_ID) — your original rulepol_ranked = (pol_ok    .withColumn("_rn", F.row_number().over(        Window.partitionBy("Policy_ID").orderBy(F.col("Policy_Start_Date").desc_nulls_last())    )))pol_silver = pol_ranked.filter(F.col("_rn") == 1).drop("_rn")# 7) Derived features (enterprise)pol_silver = pol_silver.withColumn(    "Policy_Duration_Days",    F.when(F.col("Policy_End_Date").isNotNull() & F.col("Policy_Start_Date").isNotNull(),           F.datediff("Policy_End_Date", "Policy_Start_Date"))     .otherwise(F.lit(None)))pol_silver = pol_silver.withColumn(    "Renewal_Conversion",    F.when(F.col("Renewal_Offered_Flag") == 1, F.col("Renewal_Accepted_Flag")).otherwise(F.lit(None)))# 8) Normalize categoricals (optional: map to enterprise dictionaries)pol_silver = (pol_silver    .withColumn("Product_Line", F.initcap("Product_Line"))    .withColumn("Channel", F.initcap("Channel")))# 9) Audit + DQ summarypol_silver = add_audit_columns(pol_silver, "bronze.policies")data_quality_summary(pol_silver, "policies_silver")

In [ ]:
#create tables dim_channel and dim_product_linespark.sql("""CREATE DATABASE IF NOT EXISTS bupa_databricks_workspace.bupa_reference""")spark.sql("""CREATE TABLE IF NOT EXISTS bupa_databricks_workspace.bupa_reference.dim_channel (  Channel STRING)USING DELTA""")spark.sql("""CREATE TABLE IF NOT EXISTS bupa_databricks_workspace.bupa_reference.dim_product_line (  Product_Line STRING)USING DELTA""")

In [ ]:
print("Exists?", spark.catalog.tableExists(f"{CATALOG_REF}.dim_channel"))# Peek ref valuesif spark.catalog.tableExists(f"{CATALOG_REF}.dim_channel"):    spark.table(f"{CATALOG_REF}.dim_channel").show(10, truncate=False)# Peek your policy channels (AFTER your normalization step)pol_silver.select("Channel").distinct().orderBy("Channel").show(20, truncate=False)

In [ ]:
# Seed the reference table to match your normalized Channel valuesspark.createDataFrame(    [("Agent",), ("Broker",), ("Online",), ("Partner",)],    "Channel STRING").write.mode("overwrite").saveAsTable(f"{CATALOG_REF}.dim_channel")# sanity checkspark.table(f"{CATALOG_REF}.dim_channel").show(truncate=False)

In [ ]:
def dq_left_anti_ref(df, df_ref, key_col: str, ref_col: str, name: str, severity: str,                     table_label: str, quarantine_fn, quarantine_code: str):    ref = df_ref.select(F.col(ref_col).alias("_ref_key")).dropDuplicates()    bad = df.join(ref, F.col(key_col) == F.col("_ref_key"), "left_anti")    bad_cnt = bad.count()    total = df.count()    if bad_cnt > 0:        print(f"❌ DQ FAIL [{table_label}] {name}: {bad_cnt}/{total} rows ({(bad_cnt/max(total,1))*100:.4f}%) missing in reference")        if quarantine_fn:            quarantine_fn(bad, quarantine_code, table_label)        if severity.lower() == "error":            raise Exception(f"DQ gate failed: {table_label} · {name}")    else:        print(f"✅ DQ PASS [{table_label}] {name}")

%md# ------

In [ ]:
from pyspark.sql import functions as F# Make sure policies are normalized the same way you plan to validatepol_silver = pol_silver.withColumn("Product_Line", F.initcap(F.trim(F.col("Product_Line"))))print("Distinct Product_Line values in policies:")pol_silver.select("Product_Line").distinct().orderBy("Product_Line").show(50, truncate=False)print("Null Product_Line count:")pol_silver.select(F.sum(F.col("Product_Line").isNull().cast("int")).alias("nulls")).show()

In [ ]:
from pyspark.sql import functions as Fpol_silver = pol_silver.withColumn("Product_Line", F.initcap(F.trim(F.col("Product_Line"))))

In [ ]:
# Overwrite the reference dictionary with what your data actually usesspark.createDataFrame(    [("Accident",), ("Dental",), ("Health",), ("Motor",), ("Travel",)],    "Product_Line STRING").write.mode("overwrite").saveAsTable(f"{CATALOG_REF}.dim_product_line")# sanity checkspark.table(f"{CATALOG_REF}.dim_product_line").show(truncate=False)

In [ ]:
# ============ POLICIES · DQ EXPECTATIONS ============#tbl = f"{CATALOG_SILVER}.policies"# 1) Keys presentdq_expect(pol_silver, "key_not_null",          "Policy_ID IS NOT NULL AND Customer_ID IS NOT NULL",          "error", "policies", quarantine, "null_key_policy_or_customer")# 2) Dates consistent (allow NULL end)dq_expect(pol_silver, "date_order",          "Policy_Start_Date IS NULL OR Policy_End_Date IS NULL OR Policy_End_Date >= Policy_Start_Date",          "error", "policies", quarantine, "date_order_violation")# 3) Money non-negativedq_expect(pol_silver, "money_non_negative",          "coalesce(Annual_Premium_GBP,0) >= 0 AND coalesce(Sum_Insured_GBP,0) >= 0",          "error", "policies", quarantine, "negative_money_values")# 4) Discount in [0,100]dq_expect(    pol_silver,    "discount_bounds",    "`Discount_Offered_%` IS NULL OR (`Discount_Offered_%` >= 0 AND `Discount_Offered_%` <= 100)",    "error",    "policies",    quarantine,    "discount_out_of_bounds")# 5) Renewal logic: accepted requires offereddq_expect(pol_silver, "renewal_consistency",          "NOT (Renewal_Accepted_Flag = 1 AND coalesce(Renewal_Offered_Flag,0) <> 1)",          "error", "policies", quarantine, "renewal_inconsistent")# 6) Reference validation (dim tables). Adjust dims as needed.# Set the correct catalog and schema for reference tablesCATALOG_REF = "bupa_databricks_workspace.bupa_reference"dim_channel       = spark.table(f"{CATALOG_REF}.dim_channel")         # columns: Channeldim_product_line  = spark.table(f"{CATALOG_REF}.dim_product_line")    # columns: Product_Linedq_left_anti_ref(pol_silver, dim_channel, "Channel", "Channel",                 "channel_in_dictionary", "error", "policies", quarantine, "invalid_channel")dq_left_anti_ref(pol_silver, dim_product_line, "Product_Line", "Product_Line",                 "product_line_in_dictionary", "error", "policies", quarantine, "invalid_product_line")

In [ ]:
MAGIC %sqlALTER TABLE ${CATALOG_SILVER}.policies DROP CONSTRAINT IF EXISTS chk_policy_product_line;ALTER TABLE ${CATALOG_SILVER}.policies ADD CONSTRAINT chk_policy_product_lineCHECK (Product_Line IN ('Accident','Dental','Health','Motor','Travel'));

In [ ]:
#spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")# Write & register the Delta table with schema evolution enabled(pol_silver.write.format("delta")  .mode("overwrite")  .option("overwriteSchema", "true")  # Enable schema evolution  .save(paths_silver["policies"]))# Create the database if it doesn't existspark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_SILVER}")# Save DataFrames as tables in the databasepol_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{CATALOG_SILVER}.policies")print("Policies → Silver complete.")

In [ ]:
policies_df = spark.sql(f"SELECT * FROM {CATALOG_SILVER}.policies")display(policies_df)